# Calculus Knowledge Graph — From OpenStax Textbook

Builds a pedagogically meaningful knowledge graph of Calculus concepts from the OpenStax Calculus Vol 1 textbook.

**Key difference from the Wikipedia graph:**
- Nodes are **concepts**, not just page titles
- Edges are **typed** (prerequisite, uses, is-part-of, related-to)
- Structure mirrors how a student actually learns calculus

**Node types:**
- **Chapter** — top-level organizational unit
- **Topic** — a section in the textbook (e.g., 3.3 Differentiation Rules)
- **Concept** — a specific idea within a topic (e.g., Product Rule, Chain Rule)

**Edge types:**
- `prerequisite` — A → B means you need A before B (directed, acyclic)
- `is_part_of` — concept belongs to a topic, topic belongs to chapter
- `uses` — B uses A as a tool but A isn't strictly a prerequisite
- `related_to` — A and B are analogous or frequently confused

**Author:** Catherine Cho — Hanlon/Coulter Scholars

In [ ]:
%pip install networkx matplotlib pyvis

In [ ]:
import networkx as nx
import json

G = nx.DiGraph()
print("Ready.")

## Step 1: Define the textbook structure (Chapters & Topics)

Directly from the OpenStax Calculus Volume 1 table of contents.
Each section becomes a **Topic** node. Each chapter becomes a **Chapter** node.

In [ ]:
# Full table of contents from OpenStax Calculus Volume 1
TEXTBOOK = {
    "Ch1: Functions and Graphs": [
        "1.1 Review of Functions",
        "1.2 Basic Classes of Functions",
        "1.3 Trigonometric Functions",
        "1.4 Inverse Functions",
        "1.5 Exponential and Logarithmic Functions",
    ],
    "Ch2: Limits": [
        "2.1 A Preview of Calculus",
        "2.2 The Limit of a Function",
        "2.3 The Limit Laws",
        "2.4 Continuity",
        "2.5 The Precise Definition of a Limit",
    ],
    "Ch3: Derivatives": [
        "3.1 Defining the Derivative",
        "3.2 The Derivative as a Function",
        "3.3 Differentiation Rules",
        "3.4 Derivatives as Rates of Change",
        "3.5 Derivatives of Trigonometric Functions",
        "3.6 The Chain Rule",
        "3.7 Derivatives of Inverse Functions",
        "3.8 Implicit Differentiation",
        "3.9 Derivatives of Exponential and Logarithmic Functions",
    ],
    "Ch4: Applications of Derivatives": [
        "4.1 Related Rates",
        "4.2 Linear Approximations and Differentials",
        "4.3 Maxima and Minima",
        "4.4 The Mean Value Theorem",
        "4.5 Derivatives and the Shape of a Graph",
        "4.6 Limits at Infinity and Asymptotes",
        "4.7 Applied Optimization Problems",
        "4.8 L'Hopital's Rule",
        "4.9 Newton's Method",
        "4.10 Antiderivatives",
    ],
    "Ch5: Integration": [
        "5.1 Approximating Areas",
        "5.2 The Definite Integral",
        "5.3 The Fundamental Theorem of Calculus",
        "5.4 Integration Formulas and the Net Change Theorem",
        "5.5 Substitution",
        "5.6 Integrals Involving Exponential and Logarithmic Functions",
        "5.7 Integrals Resulting in Inverse Trigonometric Functions",
    ],
    "Ch6: Applications of Integration": [
        "6.1 Areas Between Curves",
        "6.2 Determining Volumes by Slicing",
        "6.3 Volumes of Revolution: Cylindrical Shells",
        "6.4 Arc Length of a Curve and Surface Area",
        "6.5 Physical Applications",
        "6.6 Moments and Centers of Mass",
        "6.7 Integrals, Exponential Functions, and Logarithms",
        "6.8 Exponential Growth and Decay",
        "6.9 Calculus of the Hyperbolic Functions",
    ],
}

# Add chapter and topic nodes + is_part_of edges
for chapter, sections in TEXTBOOK.items():
    G.add_node(chapter, type="chapter")
    for section in sections:
        G.add_node(section, type="topic")
        G.add_edge(section, chapter, edge_type="is_part_of")

print(f"Chapters: {sum(1 for _, d in G.nodes(data=True) if d.get('type') == 'chapter')}")
print(f"Topics: {sum(1 for _, d in G.nodes(data=True) if d.get('type') == 'topic')}")

## Step 2: Add sequential prerequisite edges between topics

The textbook ordering itself is a signal: section 3.6 assumes you've read 3.1–3.5.
We add a `prerequisite` edge from each section to the one immediately after it.
This gives us the backbone DAG.

In [ ]:
# Sequential ordering within chapters
for chapter, sections in TEXTBOOK.items():
    for i in range(len(sections) - 1):
        G.add_edge(sections[i], sections[i+1], edge_type="prerequisite", source="textbook_order")

# Cross-chapter prerequisites (last section of ch N → first section of ch N+1)
chapters = list(TEXTBOOK.keys())
for i in range(len(chapters) - 1):
    last_section = TEXTBOOK[chapters[i]][-1]
    first_section = TEXTBOOK[chapters[i+1]][0]
    G.add_edge(last_section, first_section, edge_type="prerequisite", source="chapter_order")

print(f"Edges so far: {G.number_of_edges()}")

## Step 3: Add concept-level nodes

This is where the pedagogical value lives. Each topic has 3–10 specific concepts
that a student either grasps or doesn't. These are extracted from the textbook's
definitions, theorems, and key ideas.

Each concept gets:
- A short `description` (what it is)
- A `hint` (what to think about when stuck — this feeds your tutoring system)

In [ ]:
CONCEPTS = {
    # === CHAPTER 1: Functions and Graphs ===
    "1.1 Review of Functions": [
        {"name": "Function definition", "desc": "A rule assigning each input to exactly one output", "hint": "Think: does every x give exactly one y?"},
        {"name": "Domain and range", "desc": "The set of valid inputs and possible outputs", "hint": "Ask: what values of x make this expression undefined?"},
        {"name": "Function composition", "desc": "Applying one function to the output of another: f(g(x))", "hint": "Work inside-out: evaluate g(x) first, then plug into f"},
        {"name": "Vertical line test", "desc": "A graph represents a function iff every vertical line crosses it at most once", "hint": "Draw vertical lines through the graph — does any hit twice?"},
        {"name": "Symmetry (even/odd functions)", "desc": "Even: f(-x)=f(x), Odd: f(-x)=-f(x)", "hint": "Replace x with -x and simplify — what do you get?"},
    ],
    "1.2 Basic Classes of Functions": [
        {"name": "Linear functions", "desc": "f(x) = mx + b, constant rate of change", "hint": "Slope = rise/run = (y2-y1)/(x2-x1)"},
        {"name": "Polynomial functions", "desc": "Sums of terms a_n*x^n; degree determines end behavior", "hint": "The highest power term dominates as x → ±∞"},
        {"name": "Rational functions", "desc": "Ratio of two polynomials p(x)/q(x)", "hint": "Undefined where denominator = 0; check for holes vs. asymptotes"},
        {"name": "Root/radical functions", "desc": "f(x) = x^(1/n); domain restricted for even roots", "hint": "Even roots need non-negative radicands"},
        {"name": "Piecewise functions", "desc": "Different formulas on different intervals", "hint": "Check which piece applies for your x value; check continuity at boundaries"},
    ],
    "1.3 Trigonometric Functions": [
        {"name": "Unit circle definition", "desc": "sin θ and cos θ as coordinates on the unit circle", "hint": "(cos θ, sin θ) is the point on the circle at angle θ"},
        {"name": "Trig identities", "desc": "sin²θ + cos²θ = 1 and derived identities", "hint": "The Pythagorean identity is the master key; divide by cos² or sin² for the others"},
        {"name": "Graphs of trig functions", "desc": "Period, amplitude, and phase shifts of sin, cos, tan", "hint": "Period of sin(bx) = 2π/b; amplitude = |a| in a·sin(x)"},
    ],
    "1.4 Inverse Functions": [
        {"name": "One-to-one functions", "desc": "Each output comes from exactly one input (passes horizontal line test)", "hint": "Horizontal line test: does any horizontal line cross the graph twice?"},
        {"name": "Finding inverse functions", "desc": "Swap x and y, then solve for y", "hint": "f(f⁻¹(x)) = x — the inverse undoes the function"},
        {"name": "Inverse trig functions", "desc": "arcsin, arccos, arctan with restricted domains", "hint": "arcsin output is always in [-π/2, π/2]; arccos in [0, π]"},
    ],
    "1.5 Exponential and Logarithmic Functions": [
        {"name": "Exponential functions", "desc": "f(x) = a^x; base e is natural exponential", "hint": "Key property: a^(x+y) = a^x · a^y"},
        {"name": "Logarithmic functions", "desc": "log_a(x) is the inverse of a^x", "hint": "log_a(x) = y means a^y = x — rewrite to convert between forms"},
        {"name": "Log rules", "desc": "log(ab) = log a + log b; log(a^n) = n·log a", "hint": "Logs turn multiplication into addition, exponents into multiplication"},
        {"name": "Natural log (ln)", "desc": "log base e; ln(e^x) = x and e^(ln x) = x", "hint": "ln and e^x are inverses — they cancel each other"},
    ],

    # === CHAPTER 2: Limits ===
    "2.1 A Preview of Calculus": [
        {"name": "Tangent line problem", "desc": "Finding the slope of a curve at a point via secant lines", "hint": "Secant slope = (f(a+h)-f(a))/h; let h → 0 to get tangent slope"},
        {"name": "Area under a curve", "desc": "Approximating area using rectangles; leads to integration", "hint": "More rectangles = better approximation; infinite rectangles = exact area"},
    ],
    "2.2 The Limit of a Function": [
        {"name": "Intuitive definition of limit", "desc": "lim_{x→a} f(x) = L means f(x) approaches L as x approaches a", "hint": "The limit is about what f(x) APPROACHES, not what f(a) actually equals"},
        {"name": "One-sided limits", "desc": "Limits from the left (x→a⁻) and right (x→a⁺)", "hint": "Two-sided limit exists only if both one-sided limits exist and are equal"},
        {"name": "Infinite limits", "desc": "f(x) → ±∞ as x → a; vertical asymptotes", "hint": "Look for division by zero — check sign from left vs. right"},
    ],
    "2.3 The Limit Laws": [
        {"name": "Limit laws (sum, product, quotient)", "desc": "Limits distribute over arithmetic operations", "hint": "lim(f+g) = lim f + lim g — but only if both limits exist"},
        {"name": "Squeeze theorem", "desc": "If g(x) ≤ f(x) ≤ h(x) and lim g = lim h = L, then lim f = L", "hint": "Trap f(x) between two functions that both converge to L"},
        {"name": "Direct substitution", "desc": "For continuous functions, lim_{x→a} f(x) = f(a)", "hint": "Try plugging in first — only use other techniques if you get 0/0 or ∞/∞"},
    ],
    "2.4 Continuity": [
        {"name": "Definition of continuity", "desc": "f is continuous at a if lim_{x→a} f(x) = f(a)", "hint": "Three checks: f(a) exists, limit exists, they're equal"},
        {"name": "Types of discontinuity", "desc": "Removable (hole), jump, infinite (asymptote)", "hint": "Removable = limit exists but ≠ f(a); jump = one-sided limits differ"},
        {"name": "Intermediate Value Theorem", "desc": "If f is continuous on [a,b] and N is between f(a) and f(b), then f(c)=N for some c in (a,b)", "hint": "Continuous functions can't skip values — they must pass through everything in between"},
    ],
    "2.5 The Precise Definition of a Limit": [
        {"name": "Epsilon-delta definition", "desc": "For every ε>0 there exists δ>0 such that |x-a|<δ implies |f(x)-L|<ε", "hint": "ε controls how close f(x) must be to L; δ controls how close x must be to a"},
    ],

    # === CHAPTER 3: Derivatives ===
    "3.1 Defining the Derivative": [
        {"name": "Derivative at a point", "desc": "f'(a) = lim_{h→0} (f(a+h)-f(a))/h", "hint": "This is the slope of the tangent line at x=a"},
        {"name": "Tangent line equation", "desc": "y - f(a) = f'(a)(x - a)", "hint": "You need two things: the point (a, f(a)) and the slope f'(a)"},
        {"name": "Differentiability vs continuity", "desc": "Differentiable ⟹ continuous, but not vice versa", "hint": "|x| is continuous everywhere but not differentiable at x=0 (corner)"},
    ],
    "3.2 The Derivative as a Function": [
        {"name": "Derivative function", "desc": "f'(x) = lim_{h→0} (f(x+h)-f(x))/h for all x", "hint": "f'(x) gives you a NEW function that outputs the slope at each point"},
        {"name": "Higher-order derivatives", "desc": "f''(x), f'''(x), etc. — derivatives of derivatives", "hint": "f'' measures how the slope is changing — it's about curvature"},
        {"name": "Notation (Leibniz vs Lagrange)", "desc": "dy/dx vs f'(x) — different notations, same idea", "hint": "dy/dx reminds you it's a ratio of tiny changes; f'(x) is more compact"},
    ],
    "3.3 Differentiation Rules": [
        {"name": "Power rule", "desc": "d/dx[x^n] = n·x^(n-1)", "hint": "Bring the exponent down, reduce it by 1"},
        {"name": "Sum/difference rule", "desc": "(f ± g)' = f' ± g'", "hint": "Differentiate each term separately"},
        {"name": "Constant multiple rule", "desc": "(c·f)' = c·f'", "hint": "Constants just come along for the ride"},
        {"name": "Product rule", "desc": "(fg)' = f'g + fg'", "hint": "Each factor gets a turn being differentiated while the other stays"},
        {"name": "Quotient rule", "desc": "(f/g)' = (f'g - fg')/g²", "hint": "Low d-high minus high d-low, over the square of what's below"},
    ],
    "3.4 Derivatives as Rates of Change": [
        {"name": "Velocity and acceleration", "desc": "v(t) = s'(t), a(t) = v'(t) = s''(t)", "hint": "Position → velocity → acceleration: each derivative is the next"},
        {"name": "Marginal cost/revenue", "desc": "The derivative of a cost function approximates the cost of one more unit", "hint": "C'(x) ≈ C(x+1) - C(x) — the cost of producing one additional item"},
    ],
    "3.5 Derivatives of Trigonometric Functions": [
        {"name": "d/dx[sin x] = cos x", "desc": "Derivative of sine is cosine", "hint": "Comes from the limit definition using sin(a+h) expansion"},
        {"name": "d/dx[cos x] = -sin x", "desc": "Derivative of cosine is negative sine", "hint": "The negative sign! cos goes DOWN from its peak at x=0"},
        {"name": "d/dx[tan x] = sec²x", "desc": "Derivative of tangent is secant squared", "hint": "Derive from tan = sin/cos using quotient rule"},
    ],
    "3.6 The Chain Rule": [
        {"name": "Chain rule", "desc": "d/dx[f(g(x))] = f'(g(x))·g'(x)", "hint": "Differentiate the OUTER function, keep the INNER unchanged, then multiply by the derivative of the INNER"},
        {"name": "Chain rule with Leibniz notation", "desc": "dy/dx = (dy/du)·(du/dx)", "hint": "Treat dy/dx like a fraction — the du's cancel"},
    ],
    "3.7 Derivatives of Inverse Functions": [
        {"name": "Inverse function theorem", "desc": "(f⁻¹)'(x) = 1/f'(f⁻¹(x))", "hint": "The slope of the inverse is the reciprocal of the original's slope"},
        {"name": "Derivatives of inverse trig functions", "desc": "d/dx[arcsin x] = 1/√(1-x²), etc.", "hint": "Use implicit differentiation: let y = arcsin(x), so sin y = x, differentiate both sides"},
    ],
    "3.8 Implicit Differentiation": [
        {"name": "Implicit differentiation", "desc": "Differentiate both sides of an equation with respect to x, using chain rule for y terms", "hint": "Every time you differentiate a y term, tack on a dy/dx"},
    ],
    "3.9 Derivatives of Exponential and Logarithmic Functions": [
        {"name": "d/dx[e^x] = e^x", "desc": "e^x is its own derivative — the defining property of e", "hint": "This is what makes e special among all bases"},
        {"name": "d/dx[ln x] = 1/x", "desc": "The derivative of natural log is the reciprocal", "hint": "Comes from e^x and ln being inverses"},
        {"name": "d/dx[a^x] = a^x · ln(a)", "desc": "General exponential: rewrite as e^(x·ln a) and use chain rule", "hint": "Rewrite any exponential in base e: a^x = e^(x ln a)"},
        {"name": "Logarithmic differentiation", "desc": "Take ln of both sides, differentiate, solve for y'", "hint": "Useful for x^x or products of many terms — ln turns products into sums"},
    ],

    # === CHAPTER 4: Applications of Derivatives ===
    "4.1 Related Rates": [
        {"name": "Related rates setup", "desc": "Write an equation relating quantities, differentiate both sides with respect to time", "hint": "Step 1: draw a picture. Step 2: find the equation relating the variables. Step 3: differentiate implicitly with respect to t."},
    ],
    "4.2 Linear Approximations and Differentials": [
        {"name": "Linear approximation", "desc": "f(x) ≈ f(a) + f'(a)(x-a) near x=a", "hint": "The tangent line IS the linear approximation — they're the same thing"},
        {"name": "Differentials", "desc": "dy = f'(x)·dx; represents approximate change in y", "hint": "dx is a small change in x; dy estimates the resulting change in y"},
    ],
    "4.3 Maxima and Minima": [
        {"name": "Critical points", "desc": "Points where f'(x) = 0 or f'(x) is undefined", "hint": "Set the derivative equal to zero and solve — but also check where f' DNE"},
        {"name": "Extreme Value Theorem", "desc": "A continuous function on [a,b] has absolute max and min", "hint": "Check critical points AND endpoints — the absolute extrema must be at one of these"},
        {"name": "First derivative test", "desc": "f' changes sign at a critical point → local extremum", "hint": "+ to - means local max; - to + means local min"},
    ],
    "4.4 The Mean Value Theorem": [
        {"name": "Mean Value Theorem", "desc": "If f is continuous on [a,b] and differentiable on (a,b), then f'(c) = (f(b)-f(a))/(b-a) for some c", "hint": "Somewhere on the curve, the tangent line is parallel to the secant line"},
        {"name": "Rolle's Theorem", "desc": "Special case of MVT: if f(a)=f(b), then f'(c)=0 for some c in (a,b)", "hint": "If a function starts and ends at the same height, it must turn around somewhere"},
    ],
    "4.5 Derivatives and the Shape of a Graph": [
        {"name": "Increasing/decreasing test", "desc": "f'>0 → increasing; f'<0 → decreasing", "hint": "Positive slope means going up; negative slope means going down"},
        {"name": "Concavity", "desc": "f''>0 → concave up; f''<0 → concave down", "hint": "Concave up = holds water (smile); concave down = spills water (frown)"},
        {"name": "Inflection points", "desc": "Points where concavity changes; f''(x) = 0 or undefined", "hint": "Where the curve switches from smile to frown or vice versa"},
        {"name": "Second derivative test", "desc": "At critical point: f''>0 → local min, f''<0 → local max", "hint": "If the curve is concave UP at a critical point, it's a valley (min)"},
    ],
    "4.6 Limits at Infinity and Asymptotes": [
        {"name": "Limits at infinity", "desc": "Behavior of f(x) as x → ±∞", "hint": "Divide numerator and denominator by the highest power of x"},
        {"name": "Horizontal asymptotes", "desc": "y = L is a horizontal asymptote if lim_{x→±∞} f(x) = L", "hint": "Compare degrees: same degree → ratio of leading coefficients"},
    ],
    "4.7 Applied Optimization Problems": [
        {"name": "Optimization strategy", "desc": "Express quantity to optimize as a single-variable function, find critical points", "hint": "Step 1: identify what to maximize/minimize. Step 2: write it as f(x). Step 3: set f'(x)=0."},
    ],
    "4.8 L'Hopital's Rule": [
        {"name": "L'Hopital's Rule", "desc": "If lim f(x)/g(x) = 0/0 or ∞/∞, then lim f/g = lim f'/g'", "hint": "Only works for indeterminate forms 0/0 or ∞/∞ — check first!"},
    ],
    "4.9 Newton's Method": [
        {"name": "Newton's Method", "desc": "x_{n+1} = x_n - f(x_n)/f'(x_n) to find roots", "hint": "Follow the tangent line to the x-axis; that's your next guess"},
    ],
    "4.10 Antiderivatives": [
        {"name": "Antiderivative", "desc": "F is an antiderivative of f if F'(x) = f(x)", "hint": "Ask: what function, when differentiated, gives me this?"},
        {"name": "General antiderivative (+C)", "desc": "All antiderivatives differ by a constant", "hint": "Always add +C — there are infinitely many antiderivatives"},
        {"name": "Basic antiderivative rules", "desc": "∫x^n dx = x^(n+1)/(n+1) + C; reverse power rule", "hint": "Raise the exponent by 1, divide by the new exponent"},
    ],

    # === CHAPTER 5: Integration ===
    "5.1 Approximating Areas": [
        {"name": "Riemann sums", "desc": "Approximate area using left, right, or midpoint rectangles", "hint": "Width = (b-a)/n for each rectangle; height = function value at chosen point"},
        {"name": "Sigma notation", "desc": "Compact notation for sums: Σ_{i=1}^{n} f(x_i)Δx", "hint": "The Σ just means add up all the terms from i=1 to i=n"},
    ],
    "5.2 The Definite Integral": [
        {"name": "Definite integral definition", "desc": "∫_a^b f(x)dx = lim_{n→∞} Σ f(x_i)Δx", "hint": "The integral IS the limit of Riemann sums as rectangles get infinitely thin"},
        {"name": "Properties of definite integrals", "desc": "Linearity, additivity over intervals, comparison properties", "hint": "∫(f+g) = ∫f + ∫g; ∫_a^b + ∫_b^c = ∫_a^c"},
    ],
    "5.3 The Fundamental Theorem of Calculus": [
        {"name": "FTC Part 1", "desc": "d/dx[∫_a^x f(t)dt] = f(x) — differentiation undoes integration", "hint": "The derivative of an accumulation function just gives you back f(x)"},
        {"name": "FTC Part 2", "desc": "∫_a^b f(x)dx = F(b) - F(a) where F is any antiderivative of f", "hint": "Find an antiderivative, plug in the bounds, subtract. That's it."},
    ],
    "5.4 Integration Formulas and the Net Change Theorem": [
        {"name": "Net change theorem", "desc": "∫_a^b F'(x)dx = F(b) - F(a): the integral of a rate gives total change", "hint": "If you integrate velocity, you get displacement; integrate rate, get total"},
    ],
    "5.5 Substitution": [
        {"name": "U-substitution", "desc": "Reverse of chain rule: let u = inner function, du = its derivative dx", "hint": "Look for a function AND its derivative in the integrand — that's your u and du"},
        {"name": "Substitution in definite integrals", "desc": "Change the bounds when you change the variable", "hint": "If u=g(x), new bounds are u=g(a) and u=g(b)"},
    ],
    "5.6 Integrals Involving Exponential and Logarithmic Functions": [
        {"name": "∫e^x dx = e^x + C", "desc": "Exponential integrates to itself", "hint": "Same as the derivative rule — e^x is its own integral too"},
        {"name": "∫(1/x)dx = ln|x| + C", "desc": "The missing case from the power rule (n = -1)", "hint": "The power rule doesn't work for x^(-1) because you'd divide by 0 — ln fills the gap"},
    ],
    "5.7 Integrals Resulting in Inverse Trigonometric Functions": [
        {"name": "∫1/√(1-x²)dx = arcsin(x)+C", "desc": "Integral that produces inverse sine", "hint": "Pattern: 1/√(a²-x²) → arcsin(x/a)"},
        {"name": "∫1/(1+x²)dx = arctan(x)+C", "desc": "Integral that produces inverse tangent", "hint": "Pattern: 1/(a²+x²) → (1/a)arctan(x/a)"},
    ],

    # === CHAPTER 6: Applications of Integration ===
    "6.1 Areas Between Curves": [
        {"name": "Area between curves", "desc": "A = ∫_a^b |f(x)-g(x)| dx = ∫(top - bottom)dx", "hint": "Always subtract: (top curve) minus (bottom curve). Find intersection points for bounds."},
    ],
    "6.2 Determining Volumes by Slicing": [
        {"name": "Disk method", "desc": "V = π∫[R(x)]² dx for solids of revolution", "hint": "Each cross-section is a disk with radius = distance from axis to curve"},
        {"name": "Washer method", "desc": "V = π∫([R(x)]²-[r(x)]²)dx for hollow solids", "hint": "Outer radius minus inner radius — like a donut cross-section"},
    ],
    "6.3 Volumes of Revolution: Cylindrical Shells": [
        {"name": "Shell method", "desc": "V = 2π∫x·f(x)dx — wrap rectangles around the axis", "hint": "Shell = 2π × radius × height × thickness. Use when disk method is hard (e.g., rotating around y-axis)"},
    ],
    "6.4 Arc Length of a Curve and Surface Area": [
        {"name": "Arc length formula", "desc": "L = ∫_a^b √(1+[f'(x)]²) dx", "hint": "Comes from Pythagorean theorem on tiny segments: ds² = dx² + dy²"},
        {"name": "Surface area of revolution", "desc": "SA = 2π∫f(x)√(1+[f'(x)]²) dx", "hint": "Like arc length, but each tiny segment traces out a ring"},
    ],
    "6.5 Physical Applications": [
        {"name": "Work as an integral", "desc": "W = ∫_a^b F(x) dx when force varies with position", "hint": "Work = force × distance, but when force varies, you integrate"},
    ],
    "6.6 Moments and Centers of Mass": [
        {"name": "Center of mass", "desc": "x̄ = M_y/m where moments are computed via integrals", "hint": "The balance point — where the region would balance on a pin"},
    ],
    "6.7 Integrals, Exponential Functions, and Logarithms": [
        {"name": "Defining ln as an integral", "desc": "ln x = ∫_1^x (1/t) dt — alternative foundation for logarithms", "hint": "This definition works even without defining e first"},
    ],
    "6.8 Exponential Growth and Decay": [
        {"name": "Exponential growth/decay model", "desc": "dy/dt = ky has solution y = y_0·e^(kt)", "hint": "k>0 is growth, k<0 is decay. Half-life: t = ln(2)/|k|"},
    ],
    "6.9 Calculus of the Hyperbolic Functions": [
        {"name": "Hyperbolic functions", "desc": "sinh x = (e^x - e^(-x))/2, cosh x = (e^x + e^(-x))/2", "hint": "Like trig functions but with exponentials instead of circles"},
    ],
}

# Add concept nodes and is_part_of edges
for topic, concepts in CONCEPTS.items():
    for c in concepts:
        G.add_node(c["name"], type="concept", description=c["desc"], hint=c["hint"])
        G.add_edge(c["name"], topic, edge_type="is_part_of")

concept_count = sum(1 for _, d in G.nodes(data=True) if d.get('type') == 'concept')
print(f"Concept nodes added: {concept_count}")
print(f"Total nodes: {G.number_of_nodes()}, Total edges: {G.number_of_edges()}")

## Step 4: Add cross-topic prerequisite and relationship edges

This is the critical step that makes this a KNOWLEDGE GRAPH rather than just a table of contents.
These are hand-specified based on the actual pedagogical dependencies in calculus.

In [ ]:
# Each tuple is (source_concept, target_concept, edge_type, reason)
# "prerequisite": source is needed before target
# "uses": target uses source as a tool
# "related_to": analogous, inverse, or commonly confused

RELATIONSHIPS = [
    # === Foundational prerequisites ===
    ("Function definition", "Intuitive definition of limit", "prerequisite", "Limits are defined on functions"),
    ("Domain and range", "Definition of continuity", "prerequisite", "Continuity requires function to be defined at the point"),
    ("Function composition", "Chain rule", "prerequisite", "Chain rule differentiates composed functions"),
    ("One-to-one functions", "Finding inverse functions", "prerequisite", "Only 1-1 functions have inverses"),
    ("Inverse trig functions", "Derivatives of inverse trig functions", "prerequisite", "Need to know arcsin/arccos before differentiating them"),
    ("Exponential functions", "d/dx[e^x] = e^x", "prerequisite", "Must understand e^x before differentiating it"),
    ("Logarithmic functions", "d/dx[ln x] = 1/x", "prerequisite", "Must understand ln before differentiating it"),
    ("Natural log (ln)", "d/dx[ln x] = 1/x", "prerequisite", "ln is what we're differentiating"),
    ("Log rules", "Logarithmic differentiation", "prerequisite", "Log differentiation uses log rules to simplify before differentiating"),
    ("Trig identities", "d/dx[sin x] = cos x", "prerequisite", "Deriving d/dx[sin x] uses sin(a+h) identity"),
    ("Trig identities", "d/dx[cos x] = -sin x", "prerequisite", "Same identity approach"),

    # === Limit → Derivative chain ===
    ("Intuitive definition of limit", "Derivative at a point", "prerequisite", "Derivative IS a limit"),
    ("One-sided limits", "Definition of continuity", "prerequisite", "Continuity requires the limit to exist (both sides)"),
    ("Limit laws (sum, product, quotient)", "Direct substitution", "prerequisite", "Direct substitution follows from limit laws"),
    ("Direct substitution", "Definition of continuity", "prerequisite", "Continuity means direct substitution works"),
    ("Squeeze theorem", "d/dx[sin x] = cos x", "uses", "Proof that lim sin(h)/h = 1 uses squeeze theorem"),
    ("Definition of continuity", "Intermediate Value Theorem", "prerequisite", "IVT requires continuity"),
    ("Definition of continuity", "Extreme Value Theorem", "prerequisite", "EVT requires continuity on closed interval"),
    ("Definition of continuity", "Mean Value Theorem", "prerequisite", "MVT requires continuity on [a,b]"),
    ("Differentiability vs continuity", "Mean Value Theorem", "prerequisite", "MVT requires differentiability on (a,b)"),
    ("Tangent line problem", "Derivative at a point", "prerequisite", "The tangent problem motivates the derivative"),
    ("Area under a curve", "Riemann sums", "prerequisite", "Area problem motivates Riemann sums"),

    # === Differentiation dependencies ===
    ("Derivative at a point", "Derivative function", "prerequisite", "Derivative function generalizes derivative at a point"),
    ("Derivative function", "Power rule", "prerequisite", "Power rule is derived from the limit definition"),
    ("Power rule", "Sum/difference rule", "prerequisite", "Usually taught together, power rule first"),
    ("Power rule", "Product rule", "prerequisite", "Power rule is the simplest case; product rule generalizes"),
    ("Product rule", "Quotient rule", "prerequisite", "Quotient rule can be derived from product rule"),
    ("Derivative function", "Higher-order derivatives", "prerequisite", "Need to differentiate before doing it again"),
    ("Chain rule", "Implicit differentiation", "prerequisite", "Implicit diff applies chain rule to y(x)"),
    ("Chain rule", "Related rates setup", "prerequisite", "Related rates = implicit diff with respect to t"),
    ("Implicit differentiation", "Related rates setup", "prerequisite", "Related rates uses implicit differentiation"),
    ("Chain rule", "U-substitution", "prerequisite", "U-sub is the chain rule in reverse"),
    ("Chain rule", "d/dx[a^x] = a^x · ln(a)", "uses", "Rewrite a^x = e^(x ln a), then chain rule"),

    # === Derivative applications dependencies ===
    ("Derivative at a point", "Tangent line equation", "prerequisite", "Tangent line uses the derivative as slope"),
    ("Tangent line equation", "Linear approximation", "prerequisite", "Linear approximation IS the tangent line"),
    ("Tangent line equation", "Newton's Method", "uses", "Newton's method follows tangent lines to roots"),
    ("Critical points", "First derivative test", "prerequisite", "Must find critical points before testing them"),
    ("Critical points", "Second derivative test", "prerequisite", "Second derivative test evaluates at critical points"),
    ("Critical points", "Optimization strategy", "prerequisite", "Optimization requires finding critical points"),
    ("Increasing/decreasing test", "First derivative test", "prerequisite", "First derivative test checks sign changes"),
    ("Higher-order derivatives", "Concavity", "prerequisite", "Concavity uses f''(x)"),
    ("Concavity", "Second derivative test", "prerequisite", "Second derivative test uses concavity"),
    ("Concavity", "Inflection points", "prerequisite", "Inflection = concavity change"),
    ("Velocity and acceleration", "Higher-order derivatives", "uses", "Acceleration is second derivative of position"),
    ("Limits at infinity", "Horizontal asymptotes", "prerequisite", "Horizontal asymptote = limit at infinity"),
    ("L'Hopital's Rule", "Limits at infinity", "uses", "L'Hopital often used for limits at infinity"),

    # === Integration chain ===
    ("Antiderivative", "FTC Part 2", "prerequisite", "FTC2 requires knowing an antiderivative"),
    ("Riemann sums", "Definite integral definition", "prerequisite", "Definite integral is the limit of Riemann sums"),
    ("Definite integral definition", "FTC Part 1", "prerequisite", "FTC1 is about the definite integral"),
    ("FTC Part 1", "FTC Part 2", "prerequisite", "FTC1 is used to prove FTC2"),
    ("FTC Part 2", "Net change theorem", "prerequisite", "Net change theorem is a direct consequence of FTC2"),
    ("FTC Part 2", "Area between curves", "prerequisite", "Area between curves requires evaluating definite integrals"),
    ("Basic antiderivative rules", "U-substitution", "prerequisite", "Need basic rules before learning substitution"),
    ("U-substitution", "Substitution in definite integrals", "prerequisite", "Apply u-sub technique to definite integrals"),
    ("d/dx[e^x] = e^x", "∫e^x dx = e^x + C", "prerequisite", "Integration rule comes from reversing derivative"),
    ("d/dx[ln x] = 1/x", "∫(1/x)dx = ln|x| + C", "prerequisite", "Integration rule from reversing derivative"),
    ("Derivatives of inverse trig functions", "∫1/√(1-x²)dx = arcsin(x)+C", "prerequisite", "Integral formula from reversing derivative of arcsin"),
    ("Derivatives of inverse trig functions", "∫1/(1+x²)dx = arctan(x)+C", "prerequisite", "Integral formula from reversing derivative of arctan"),

    # === Integration applications ===
    ("Area between curves", "Disk method", "prerequisite", "Disk method extends area computation to 3D"),
    ("Disk method", "Washer method", "prerequisite", "Washer = disk with a hole"),
    ("Disk method", "Shell method", "related_to", "Alternative method for same type of problem"),
    ("FTC Part 2", "Work as an integral", "prerequisite", "Work computation requires evaluating integrals"),
    ("FTC Part 2", "Arc length formula", "prerequisite", "Arc length is a definite integral"),
    ("Arc length formula", "Surface area of revolution", "prerequisite", "Surface area extends arc length to 3D"),
    ("d/dx[e^x] = e^x", "Exponential growth/decay model", "prerequisite", "Growth/decay ODE uses exponential derivative"),
    ("Antiderivative", "Exponential growth/decay model", "prerequisite", "Solving the ODE requires antidifferentiation"),
    ("Exponential functions", "Hyperbolic functions", "prerequisite", "Hyperbolic functions are defined via e^x"),

    # === Cross-concept analogies ("fire the neurons" connections) ===
    ("Chain rule", "U-substitution", "related_to", "U-sub is the chain rule run backwards"),
    ("Product rule", "Quotient rule", "related_to", "Quotient rule follows from product rule"),
    ("d/dx[sin x] = cos x", "d/dx[cos x] = -sin x", "related_to", "Parallel derivative rules with sign change"),
    ("Power rule", "Basic antiderivative rules", "related_to", "Anti-differentiation is the reverse of the power rule"),
    ("Disk method", "Shell method", "related_to", "Two methods for the same type of volume problem"),
    ("First derivative test", "Second derivative test", "related_to", "Two approaches to classifying critical points"),
    ("Derivative at a point", "Definite integral definition", "related_to", "Both defined as limits — one of a difference quotient, one of Riemann sums"),
    ("Tangent line problem", "Area under a curve", "related_to", "The two fundamental problems of calculus — connected by FTC"),
    ("Velocity and acceleration", "Net change theorem", "related_to", "Integrating velocity gives displacement — net change in action"),
    ("Linear approximation", "Differentials", "related_to", "Two perspectives on the same idea of local linearity"),
    ("Mean Value Theorem", "Rolle's Theorem", "related_to", "Rolle's is MVT with f(a)=f(b)"),
    ("Extreme Value Theorem", "Optimization strategy", "uses", "EVT guarantees that an optimum exists"),
    ("Intermediate Value Theorem", "Newton's Method", "uses", "IVT guarantees a root exists; Newton's finds it"),
    ("Sigma notation", "Riemann sums", "prerequisite", "Riemann sums are written in sigma notation"),
    ("∫(1/x)dx = ln|x| + C", "Defining ln as an integral", "related_to", "Two perspectives: ln as inverse of e^x vs. ln defined by integral"),
    ("Inverse function theorem", "Derivatives of inverse trig functions", "uses", "Inverse function theorem gives the formula structure"),
    ("Logarithmic differentiation", "d/dx[a^x] = a^x · ln(a)", "related_to", "Both use ln to handle variable exponents"),
    ("Epsilon-delta definition", "Intuitive definition of limit", "related_to", "Formal vs informal — same concept, different rigor levels"),
]

# Add all relationship edges
for src, tgt, etype, reason in RELATIONSHIPS:
    if src in G and tgt in G:
        G.add_edge(src, tgt, edge_type=etype, reason=reason)
    else:
        missing = [x for x in [src, tgt] if x not in G]
        print(f"WARNING: missing node(s): {missing}")

print(f"\nFinal graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

# Count by edge type
from collections import Counter
edge_types = Counter(d.get('edge_type', 'unknown') for _, _, d in G.edges(data=True))
for etype, count in edge_types.items():
    print(f"  {etype}: {count}")

## Step 5: Sanity checks

In [ ]:
# Check for cycles in the prerequisite subgraph (should be a DAG!)
prereq_edges = [(u, v) for u, v, d in G.edges(data=True) if d.get('edge_type') == 'prerequisite']
prereq_G = nx.DiGraph(prereq_edges)

if nx.is_directed_acyclic_graph(prereq_G):
    print("Prerequisite subgraph is a valid DAG (no cycles).")
else:
    cycles = list(nx.simple_cycles(prereq_G))
    print(f"WARNING: Found {len(cycles)} cycle(s) in prerequisites!")
    for c in cycles[:5]:
        print(f"  {' → '.join(c)}")

In [ ]:
# Most connected concepts (by total degree)
concepts_only = [n for n, d in G.nodes(data=True) if d.get('type') == 'concept']
concept_degree = sorted([(n, G.degree(n)) for n in concepts_only], key=lambda x: x[1], reverse=True)

print("Most connected concepts:")
for name, deg in concept_degree[:15]:
    in_d = G.in_degree(name)
    out_d = G.out_degree(name)
    print(f"  {deg:3d} (in:{in_d:2d} out:{out_d:2d})  {name}")

In [ ]:
# Query: What are all prerequisites for "U-substitution"?
def get_all_prerequisites(G, concept, edge_type="prerequisite"):
    """Follow prerequisite edges backward from a concept."""
    prereqs = set()
    queue = [concept]
    while queue:
        current = queue.pop(0)
        for pred in G.predecessors(current):
            edge_data = G.edges[pred, current]
            if edge_data.get("edge_type") == edge_type and pred not in prereqs:
                prereqs.add(pred)
                queue.append(pred)
    return prereqs

prereqs = get_all_prerequisites(G, "U-substitution")
print("All prerequisites for U-substitution:")
for p in sorted(prereqs):
    print(f"  ← {p}")

In [ ]:
# Query: What concepts should I review if I'm stuck on "Implicit differentiation"?
def get_study_suggestions(G, concept):
    """Get all connected concepts with their relationship type — the 'fire neurons' function."""
    suggestions = {"prerequisites": [], "related": [], "uses": []}

    # Look backward (predecessors with prerequisite/uses edges)
    for pred in G.predecessors(concept):
        edge = G.edges[pred, concept]
        etype = edge.get("edge_type", "")
        reason = edge.get("reason", "")
        if etype == "prerequisite":
            suggestions["prerequisites"].append((pred, reason))
        elif etype == "uses":
            suggestions["uses"].append((pred, reason))
        elif etype == "related_to":
            suggestions["related"].append((pred, reason))

    # Also check outgoing related_to edges
    for succ in G.successors(concept):
        edge = G.edges[concept, succ]
        if edge.get("edge_type") == "related_to":
            suggestions["related"].append((succ, edge.get("reason", "")))

    return suggestions

target = "Implicit differentiation"
node_data = G.nodes[target]
print(f"=== {target} ===")
print(f"Description: {node_data.get('description', 'N/A')}")
print(f"Hint: {node_data.get('hint', 'N/A')}")

suggestions = get_study_suggestions(G, target)
print(f"\nPrerequisites you should review:")
for name, reason in suggestions["prerequisites"]:
    print(f"  ← {name}: {reason}")
    print(f"     Hint: {G.nodes[name].get('hint', '')}")

print(f"\nRelated concepts to explore:")
for name, reason in suggestions["related"]:
    print(f"  ↔ {name}: {reason}")

## Step 6: Save everything

In [ ]:
# Save as GraphML for Gephi
nx.write_graphml(G, "calculus_kg.graphml")

# Save as JSON for the web front-end / LLM later
data = nx.node_link_data(G, edges="edges")
with open("calculus_kg.json", "w") as f:
    json.dump(data, f, indent=2)

print("Saved: calculus_kg.graphml, calculus_kg.json")

## Step 7: Interactive visualization

This one should load fine — we have ~100 concepts, not 59,000 edges.

In [ ]:
from pyvis.network import Network

# Color by node type
COLOR_MAP = {
    "chapter": "#e74c3c",   # red
    "topic": "#3498db",     # blue
    "concept": "#2ecc71",   # green
}

net = Network(height="900px", width="100%", directed=True, notebook=False, cdn_resources="in_line")

for node, data in G.nodes(data=True):
    ntype = data.get("type", "concept")
    color = COLOR_MAP.get(ntype, "#95a5a6")
    size = {"chapter": 30, "topic": 20, "concept": 12}.get(ntype, 10)
    title = f"{node}\n{data.get('description', '')}\nHint: {data.get('hint', '')}"
    net.add_node(node, label=node, color=color, size=size, title=title)

EDGE_COLORS = {
    "prerequisite": "#e74c3c",   # red
    "is_part_of": "#bdc3c7",     # gray
    "uses": "#f39c12",           # orange
    "related_to": "#9b59b6",     # purple
}

for u, v, data in G.edges(data=True):
    etype = data.get("edge_type", "unknown")
    color = EDGE_COLORS.get(etype, "#bdc3c7")
    width = 2 if etype == "prerequisite" else 1
    net.add_edge(u, v, color=color, width=width, title=f"{etype}: {data.get('reason', '')}")

net.force_atlas_2based()
net.show_buttons(filter_=['physics'])
net.save_graph("calculus_kg.html")
print("Saved: calculus_kg.html — open in browser")
print("\nLegend:")
print("  Red nodes = Chapters | Blue = Topics | Green = Concepts")
print("  Red edges = Prerequisites | Purple = Related | Orange = Uses | Gray = Part-of")

## Next steps

1. **Enrich with LLM** — For each concept pair that shares a topic, use Claude API to ask:\n   "Is A a prerequisite for B? Does B use A? Are they analogous? Or unrelated?"\n   This fills in edges we may have missed.

2. **Parse OpenStax back-references** — Scrape each section page and find phrases like\n   "recall from Section 2.3" or "as we saw in" — these are explicit prerequisite signals.

3. **Add problem types** — Map homework problems to the concepts they require.\n   This is what makes the tutoring system work: student gets stuck on problem → graph tells them what to review.

4. **Korean curriculum overlay** — During your winterim fieldwork, extract the concept ordering\n   from Korean textbooks and add as a parallel set of "prerequisite" edges with source=korean_curriculum.\n   Compare with the OpenStax ordering.

5. **LLM integration** — Use the graph as a retrieval backbone:\n   Student asks a question → identify which concept → retrieve prerequisites + hints from graph → LLM generates guidance without giving the answer.